In [4]:
from google.cloud import bigquery
from datetime import date, timedelta

client = bigquery.Client(project="prd-izipay-data-operation")

def cargar_historico_mcfs005al(var_fecha_ini: date, var_fecha_fin: date):
    var_project_operation = "prd-izipay-data-operation"
    var_project_storage   = "prd-izipay-data-storage-pv"
    var_project_sensitive = "prd-izipay-data-sensitive"
    var_dataset_bq_omni   = "bq_omni_izipay_azure_saizipaydatamarts"
    var_table             = "mcfs005al"
    var_connection        = "azure-eastus2.bq-omni-izipay-azure-saizipaydatamarts"

    # Ruta BASE sin año/mes -> eso se arma en automático por cada fecha
    var_ruta_base = "azure://saizipaydatamarts.blob.core.windows.net/ingesta-gcp/Raw/As-400/Transaccional/mcfs005al/"

    location_ext_table = "azure-eastus2"   # donde vive la tabla externa
    location_raw       = "US"              # donde vive raw_stage_as400 / raw_as400

    dias = [var_fecha_ini + timedelta(days=i) for i in range((var_fecha_fin - var_fecha_ini).days + 1)]

    for i in range(0, len(dias), 7):
        bloque = dias[i:i+7]

        # Arma cada URI usando el año/mes propio de CADA fecha (aunque el bloque cruce de mes)
        uris = ",".join(
            f'"{var_ruta_base}{d.year}/{d.month:02d}/{var_table}_{d.strftime("%Y%m%d")}.parquet"'
            for d in bloque
        )

        # 1) Crear tabla externa -> job en azure-eastus2
        query_ext = f"""
        CREATE OR REPLACE EXTERNAL TABLE `{var_project_operation}.{var_dataset_bq_omni}.{var_table}`
        WITH CONNECTION `{var_connection}`
        OPTIONS (
          format = 'PARQUET',
          uris = [{uris}]
        )
        """
        print(f"Creando tabla externa bloque {bloque[0]} a {bloque[-1]} ...")
        client.query(query_ext, location=location_ext_table).result()

        # 2) Cargar a raw -> job en US
        query_call = f"""
        CALL `{var_project_operation}.raw_stage_as400.prc_load_as400_mcfs005al_bloque`(
          '{var_project_operation}', '{var_project_storage}', '{var_project_sensitive}'
        )
        """
        print(f"Cargando a raw bloque {bloque[0]} a {bloque[-1]} ...")
        client.query(query_call, location=location_raw).result()

    print("✅ Carga histórica completa.")


# Ejemplo de uso -> ahora solo mandas las fechas, sin la ruta con año/mes:
cargar_historico_mcfs005al(
    var_fecha_ini=date(2026, 3, 1),
    var_fecha_fin=date(2026, 3, 31),
)

Creando tabla externa bloque 2026-03-01 a 2026-03-07 ...
Cargando a raw bloque 2026-03-01 a 2026-03-07 ...
Creando tabla externa bloque 2026-03-08 a 2026-03-14 ...
Cargando a raw bloque 2026-03-08 a 2026-03-14 ...
Creando tabla externa bloque 2026-03-15 a 2026-03-21 ...
Cargando a raw bloque 2026-03-15 a 2026-03-21 ...
Creando tabla externa bloque 2026-03-22 a 2026-03-28 ...
Cargando a raw bloque 2026-03-22 a 2026-03-28 ...
Creando tabla externa bloque 2026-03-29 a 2026-03-31 ...
Cargando a raw bloque 2026-03-29 a 2026-03-31 ...
✅ Carga histórica completa.
